In [2]:
from os.path import exists
from pathlib import Path
import uuid
from red_gym_env import RedGymEnv
from stable_baselines3 import A2C, PPO
from stable_baselines3.common import env_checker
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.callbacks import CheckpointCallback

In [4]:
from pyboy import PyBoy

In [5]:
import os
import pdb

starting_states = ['states\\fast_text_start.state','states\init.state', 'states\has_pokedex.state', 'states\has_pokedex_nballs.state', 'PokemonRed.gb.state']

# Initialize the emulator
pyboy = PyBoy('./PokemonRed.gb')

frame_count = 0
pressed_keys = set()
state = {'running': True}
tilemaps = {}
screen = {}

map_headers = {
    'x': 0, 'y': 0, 'map_n': 0,
    'maps_tileset': 0, 'maps_height': 0, 'maps_width': 0,
    'maps_data': 0, 'maps_text_pointer_table': 0, 'maps_level_script_pointer': 0,
    'maps_connection_byte': 0, 'maps_1st_connection_data': 0, 'maps_2nd_connection_data': 0,
    'maps_3rd_connection_data': 0, 'maps_4th_connection_data': 0,
    'map_area': 0
    }
tileset_headers = {
    'tileset bank': 0, 'pointer to blocks': 0, 'pointer to gfx': 0,
    'pointer to collision data': 0, 'talking-over tiles': 0, 'grass tile': 0,
    'pointer to collision data2': 0, 'collision_bytes': 0, 'collision_data': 0, 
    'collision_data2': 0,
}
def set_map_headers():
    map_headers['y'] = read_m(0xD361)
    map_headers['x'] = read_m(0xD362)
    map_headers['map_n'] = read_m(0xD35E)
    map_headers['maps_tileset'] = read_m(0xD367)
    map_headers['maps_height'] = read_m(0xD368) * 2
    map_headers['maps_width'] = read_m(0xD369) * 2
    map_headers['maps_data'] = read_m(0xD36A) + 256 * read_m(0xD36B)
    map_headers['maps_text_pointer_table'] = read_m(0xD36C) + 256 * read_m(0xD36D)
    map_headers['maps_level_script_pointer'] = read_m(0xD36E) + 256 * read_m(0xD36F)
    map_headers['maps_connection_byte'] = read_m(0xD370)
    map_headers['maps_1st_connection_data'] = read_m(0xD371) + 256 * read_m(0xD372) + 256 * 256 * read_m(0xD373)
    map_headers['maps_2nd_connection_data'] = read_m(0xD374) + 256 * read_m(0xD375) + 256 * 256 * read_m(0xD376)
    map_headers['maps_3rd_connection_data'] = read_m(0xD377) + 256 * read_m(0xD378) + 256 * 256 * read_m(0xD379)
    map_headers['maps_4th_connection_data'] = read_m(0xD37A) + 256 * read_m(0xD37B) + 256 * 256 * read_m(0xD37C)
    map_headers['map_area'] = (map_headers['maps_height'] * map_headers['maps_width'])

def set_tileset_headers():
    set_map_headers()
    tileset_headers['tileset bank'] = read_m(0xD52B)
    tileset_headers['pointer to blocks'] = read_m(0xD52C) + 256 * read_m(0xD52D)
    tileset_headers['pointer to gfx'] = read_m(0xD52E) + 256 * read_m(0xD52F)
    tileset_headers['pointer to collision data'] = read_m(0xD530) + 256 * read_m(0xD531)
    tileset_headers['collision data'] = pyboy.memory[tileset_headers['tileset bank'], tileset_headers['pointer to collision data']:tileset_headers['pointer to collision data']+tileset_headers['collision_bytes']]
    tileset_headers['talking-over tiles'] = read_m(0xD532) + 256 * read_m(0xD533) + 256 * 256 * read_m(0xD534)
    tileset_headers['grass tile'] = read_m(0xD535)
    tileset_headers['collision_bytes'] = 96
    tileset_headers['walkable_tiles'] = read_collision_data(tileset_headers['pointer to collision data'])

In [7]:
def read_m(addr):
    return pyboy.memory[addr]

In [6]:

for _ in range(60):
    pyboy.tick()
pyboy.stop()

In [2]:
game_area = [
    [272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272],
    [272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272],
    [272, 272, 272, 272, 288, 289, 288, 289, 288, 289, 288, 289, 288, 289, 288, 289, 272, 272, 272, 272],
    [272, 272, 272, 272, 306, 307, 306, 307, 306, 307,  12,  13, 306, 307, 306, 307, 272, 272, 272, 272],
    [272, 272, 272, 272, 288, 289, 273, 257, 273, 257,  14,  15, 273, 257, 288, 289, 272, 272, 272, 272],
    [272, 272, 272, 272, 306, 307, 257, 273, 257, 273, 257, 273, 257, 273, 306, 307, 272, 272, 272, 272],
    [272, 272, 272, 272, 273, 257, 263, 265, 265, 265, 265, 265, 265, 264, 273, 257, 272, 272, 272, 272],
    [272, 272, 272, 272, 257, 273, 306, 307,   9,   8, 306, 307, 306, 307, 257, 273, 272, 272, 272, 272],
    [272, 272, 272, 272, 350, 350, 273, 257,  11,  10, 273, 257, 273, 257, 350, 350, 272, 272, 272, 272],
    [272, 272, 272, 272, 312, 312, 257, 273, 257, 273, 257, 273, 257, 273, 312, 312, 272, 272, 272, 272],
    [272, 272, 272, 272, 312, 312, 273, 257, 273, 257, 273, 257, 273, 257, 312, 312, 272, 272, 272, 272],
    [272, 272, 272, 272, 350, 350, 257, 273, 257, 273, 257, 273, 257, 273, 350, 350, 272, 272, 272, 272],
    [272, 272, 272, 272, 273, 257, 263, 265, 265, 265, 265, 265, 265, 264, 273, 257, 272, 272, 272, 272],
    [272, 272, 272, 272, 257, 273, 306, 307, 306, 307, 306, 307, 306, 307, 257, 273, 272, 272, 272, 272],
    [272, 272, 272, 272, 288, 289, 273, 257, 273, 257, 273, 257, 273, 257, 288, 289, 272, 272, 272, 272],
    [272, 272, 272, 272, 306, 307, 257, 273, 257, 273, 257, 273, 257, 273, 306, 307, 272, 272, 272, 272],
    [272, 272, 272, 272, 288, 289, 288, 289, 288, 289, 288, 289, 288, 289, 288, 289, 272, 272, 272, 272],
    [272, 272, 272, 272, 306, 307, 306, 307, 306, 307, 306, 307, 306, 307, 306, 307, 272, 272, 272, 272]
]

In [6]:
for lst in game_area:
    print(len(lst))

20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20


In [3]:
def make_env(rank, env_conf, seed=0):
    def _init():
        env = RedGymEnv(env_conf)
        env.reset(seed=(seed + rank))
        return env
    set_random_seed(seed)
    return _init

In [16]:
ep_length = 2048 * 8
sess_path = Path(f'session_{str(uuid.uuid4())[:8]}')

In [20]:
env_config = {
            'headless': False, 'save_final_state': True, 'early_stop': False,
            'action_freq': 24, 'init_state': './has_pokedex_nballs.state', 'max_steps': ep_length, 
            'print_rewards': True, 'save_video': False, 'fast_video': True, 'session_path': sess_path,
            'gb_path': './PokemonRed.gb', 'debug': False, 'sim_frame_dist': 2_000_000.0, 
            'use_screen_explore': True, 'extra_buttons': False
        }

In [21]:
num_cpu = 1 #64 #46  # Also sets the number of episodes per training iteration

In [22]:
env = SubprocVecEnv([make_env(i, env_config) for i in range(num_cpu)])

In [23]:
checkpoint_callback = CheckpointCallback(save_freq=ep_length, save_path=sess_path,
                                    name_prefix='poke')

In [24]:
learn_steps = 40
file_name = 'session_e41c9eff/poke_38207488_steps' #'session_e41c9eff/poke_250871808_steps'

In [25]:
if exists(file_name + '.zip'):
    print('\nloading checkpoint')
    model = PPO.load(file_name, env=env)
    model.n_steps = ep_length
    model.n_envs = num_cpu
    model.rollout_buffer.buffer_size = ep_length
    model.rollout_buffer.n_envs = num_cpu
    model.rollout_buffer.reset()
else:
    model = PPO('CnnPolicy', env, verbose=1, n_steps=ep_length, batch_size=512, n_epochs=1, gamma=0.999)

for i in range(learn_steps):
    model.learn(total_timesteps=(ep_length)*num_cpu*1000, callback=checkpoint_callback)

Using cpu device
Wrapping the env in a VecTransposeImage.


In [5]:
if __name__ == '__main__':

    ep_length = 2048 * 8
    sess_path = Path(f'session_{str(uuid.uuid4())[:8]}')

    env_config = {
                'headless': False, 'save_final_state': True, 'early_stop': False,
                'action_freq': 24, 'init_state': './has_pokedex_nballs.state', 'max_steps': ep_length, 
                'print_rewards': True, 'save_video': False, 'fast_video': True, 'session_path': sess_path,
                'gb_path': './PokemonRed.gb', 'debug': False, 'sim_frame_dist': 2_000_000.0, 
                'use_screen_explore': True, 'extra_buttons': False
            }
    
    
    num_cpu = 1 #64 #46  # Also sets the number of episodes per training iteration
    env = SubprocVecEnv([make_env(i, env_config) for i in range(num_cpu)])
    
    checkpoint_callback = CheckpointCallback(save_freq=ep_length, save_path=sess_path,
                                     name_prefix='poke')
    #env_checker.check_env(env)
    learn_steps = 40
    file_name = 'session_e41c9eff/poke_38207488_steps' #'session_e41c9eff/poke_250871808_steps'
    
    #'session_bfdca25a/poke_42532864_steps' #'session_d3033abb/poke_47579136_steps' #'session_a17cc1f5/poke_33546240_steps' #'session_e4bdca71/poke_8945664_steps' #'session_eb21989e/poke_40255488_steps' #'session_80f70ab4/poke_58982400_steps'
    if exists(file_name + '.zip'):
        print('\nloading checkpoint')
        model = PPO.load(file_name, env=env)
        model.n_steps = ep_length
        model.n_envs = num_cpu
        model.rollout_buffer.buffer_size = ep_length
        model.rollout_buffer.n_envs = num_cpu
        model.rollout_buffer.reset()
    else:
        model = PPO('CnnPolicy', env, verbose=1, n_steps=ep_length, batch_size=512, n_epochs=1, gamma=0.999)
    
    for i in range(learn_steps):
        model.learn(total_timesteps=(ep_length)*num_cpu*1000, callback=checkpoint_callback)


Using cpu device
Wrapping the env in a VecTransposeImage.


In [28]:
# This code block will read the binary file and print the data in bytes
# Opening the binary file in binary mode as rb(read binary)
f = open("PokemonRed.gb.state", mode="rb")
 
# Reading file data with read() method
data = f.read()
 
# Knowing the Type of our data
print(type(data))
 
# Printing our byte sequenced data 
print(data)
 
# Closing the opened file
f.close()

<class 'bytes'>
b'\t\x00\x00\x00\x00\x01\xa0\x14\xf0\x00\x00\xfe\xc2\xfd\xdf\xb3 \x01\x01\x00\r\x00\x01\x07\x07\x08\x0f\x10\x1f\x10\x1f;<?7\x7fP\x7fB\xe0\xe0\x10\xf0\x08\xf8\x08\xf8\xdc<\xfc\xec\xfe\n\xfeB?2>9\x7fO\x7fO9?\x16\x1f\x11\x1f\x0e\x0e\xfcL|\x9c\xfe\xf2\xfe\xf2\x9c\xfch\xf8\x88\xf8pp\x07\x07\x08\x0f\x10\x1f\x10\x1f0?8?\x7f_\x7fO\xe0\xe0\x10\xf0\x08\xf8\x08\xf8\x0c\xfc\x1c\xfc\xfe\xfa\xfe\xf2?3<?{_y^<?\x17\x1f\x11\x1f\x0e\x0e\xfc\xcc<\xfc\xde\xfa\x9ez<\xfc\xe8\xf8\x88\xf8pp\x07\x07\x08\x0f\x10\x1f87|C1?\x1f\x14\x1f\x14\xe0\xe0\x10\xf0\x08\xf8\x08\xf8\x1c\xfc\xfc\xfc\xfc\xfc\xf8\x98\x1f\x10\x0b\x0c\x07\x07\x03\x03\x03\x03\x04\x07\x04\x07\x03\x03\xf0\x10\xe8x\xc8\xf8\xe88\xe88\xf0\xf0 \xe0\xc0\xc0\x03\x03|\x7f@\x7f"?4?}_\x7fPoR\xc0\xc00\xf0\x08\xf8D\xfc\xc4\xfc\xfe:\xfe\n\xf6J?2>9\x7fO\x7fK?9\x17\x1c\x11\x1f\x0e\x0e\xfcL|\x9c\xfe\xf2\xfe\xd2\xfc\x9c\xe88\x88\xf8pp\x03\x03\x0c\x0f\x10\x1f ? ?t_x_o_\xc0\xc0>\xfe\x02\xfe\x04\xfc\x0c\xfc\x1e\xfa\x1e\xfa\xf6\xfa\x7ft?;~Y\x7f\\?;\x17\

In [ ]:
# This code block will compare 2 binary files and print the difference
# Open the first save state
with open("./states/PokemonRed.gb.state", mode="rb") as f1:
    data1 = f1.read()

# Open the second save state
with open("./states/PokemonRed.gb2.state", mode="rb") as f2:
    data2 = f2.read()

# Compare the two save states
for i in range(min(len(data1), len(data2))):
    if data1[i]+1 == data2[i]:
        # print(f'Difference at byte {hex(i)}: {data1[i]} vs {data2[i]}')
        print(f'{i},')

In [14]:
pyboy = Pyboy()
pyboy.read(save_state="PokemonRed.gb.state")

In [16]:
# Open the save state file in binary mode
with open("PokemonRed.gb.state", mode="r+b") as f:
    # Seek to the position of the X coordinate
    for change_location in change_locations:
        f.seek(change_location)

        # Read the current X coordinate
        old_x = f.read(1)[0]
        print('old x', old_x)
        # Calculate the new X coordinate
        new_x = (old_x + 1) % 256
        print('attempting to set new x value to ', new_x)
        # Seek back to the position of the X coordinate
        f.seek(change_location)

        # Write the new X coordinate
        f.write(bytes([new_x]))
        f.seek(change_location)
        checked_new_x_value = f.read(1)[0]
        print('new_x_value', checked_new_x_value)

old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
attempting to set new x value to  1
checked_new_x_value 1
old x 0
at

In [11]:
for i in range(0xD9AC, 0xD9B6+1):
    print(f'{hex(i)},')

0xd9ac,
0xd9ad,
0xd9ae,
0xd9af,
0xd9b0,
0xd9b1,
0xd9b2,
0xd9b3,
0xd9b4,
0xd9b5,
0xd9b6,


In [28]:
def get_byte_locations(input1, input2):
    # D8B0-D8B1
    output_string = ''
    for count, i in enumerate(range(input1, input2+1)):
        if count < len(range(input1, input2)):
            output_string += (f'{hex(i)},')
        elif count == len(range(input1, input2+1))-1:
            output_string += (f'{hex(i)}')
    print(output_string)
            
get_byte_locations(0xD8B0, 0xD8B1)

0xd8b0,0xd8b1


In [22]:
for count, i in enumerate(range(0xD8B0, 0xD8B1+1)):
    print(f'{count} {hex(i)}')

0 0xd8b0
1 0xd8b1


In [32]:
[print(hex(i)) for i in range(0xd273, 0xD27D+1)]

0xd273
0xd274
0xd275
0xd276
0xd277
0xd278
0xd279
0xd27a
0xd27b
0xd27c
0xd27d


[None, None, None, None, None, None, None, None, None, None, None]

In [3]:
def press_key(key):
	cmd = "osascript -e 'tell application \"System Events\" to keystroke \"" + key + "\"'"
	os.system(cmd)

In [4]:
collision_data = [22, 32, 32, 33, 33, 32, 32, 33, 33, 32, 32, 2, 3, 32, 32, 18, 19, 32, 32, 32, 32, 32, 32, 32, 32, 47, 47, 32, 32, 34, 34, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 47, 47, 32, 32, 34, 34, 32, 32, 32, 32, 32, 32, 32, 32, 32, 4, 41, 41, 4, 49, 5, 5, 12, 13, 12, 13, 28, 29, 28, 29, 41, 41, 41, 41, 5, 5, 5, 5, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 43, 44, 32, 32, 45, 46, 12, 13, 32, 32, 28, 29, 32, 32, 12, 13, 32, 32, 28, 29, 32, 32, 12, 13, 12, 13, 28, 29, 28, 29, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 7, 32, 32, 32, 23, 7, 32, 32, 23, 23, 2, 3, 23, 23, 18, 19, 23, 23, 2, 3, 23, 23, 18, 19, 7, 32, 2, 3, 23, 7, 18, 19, 23, 23, 2, 3, 23, 23, 18, 19, 12, 13, 2, 3, 28, 29, 18, 19, 2, 3, 2, 3, 18, 19, 18, 19, 23, 23, 32, 32, 23, 23, 32]

2

In [4]:
pkmon1_trainer_ids = [
0xd9ac,
0xd9ad,
0xd9ae,
0xd9af,
0xd9b0,
0xd9b1,
0xd9b2,
0xd9b3,
0xd9b4,
0xd9b5
]

In [11]:
filename = 'PokemonRed.gb'

def find_bytes(filename):
    first_byte = 195
    second_byte = 165
    with open(filename, 'rb') as f:
        prev_byte = f.read(1)
        while (byte := f.read(1)):
            if ord(prev_byte) == first_byte and ord(byte) == second_byte:
                print(f"Found {first_byte} followed by {second_byte} at position {f.tell() - 2}")
                subsequent_bytes = f.read(4)
                print("Subsequent 4 bytes:", [b for b in subsequent_bytes])
            prev_byte = byte

# Call the function with the path to your .gb file
find_bytes(filename)

Found 195 followed by 165 at position 31474
Subsequent 4 bytes: [122, 150, 136, 147]
Found 195 followed by 165 at position 228273
Subsequent 4 bytes: [195, 165, 195, 165]
Found 195 followed by 165 at position 228279
Subsequent 4 bytes: [195, 165, 195, 165]
Found 195 followed by 165 at position 228285
Subsequent 4 bytes: [195, 165, 0, 0]


In [1]:
text_data = [222, 65, 172, 88, 168, 88, 0, 106, 89, 205, 60, 60, 201, 184, 88, 219, 88, 254, 88, 11, 89, 24, 89, 101, 89, 8, 250, 224, 215, 203, 127, 32, 8, 33, 209, 88, 205, 73, 60, 24, 6, 33, 214, 88, 205, 73, 60, 195, 215, 36, 23, 141, 94, 38, 80, 23, 186, 94, 38, 80, 8, 250, 224, 215, 203, 127, 32, 8, 33, 244, 88, 205, 73, 60, 24, 6, 33, 249, 88, 205, 73, 60, 195, 215, 36, 23, 238, 94, 38, 80, 23, 75, 95, 38, 80, 23, 114, 95, 38, 8]

In [100]:
start = int('C3A0', 16)  # Convert start hex to decimal
end = int('C507', 16)  # Convert end hex to decimal

# Calculate the number of values in the range
num_values = end - start + 1  # +1 to include the end value itself
num_values

360

In [97]:
byte_to_char_map = {
    0x4F: ' ',
    0x57: '#',
    0x51: '*',
    0x52: 'A1',
    0x53: 'A2',
    0x54: 'POKé',
    0x55: '+',
    0x58: '$',
    0x75: '…',
    0x7F: ' ',
    0x80: 'A',
    0x81: 'B',
    0x82: 'C',
    0x83: 'D',
    0x84: 'E',
    0x85: 'F',
    0x86: 'G',
    0x87: 'H',
    0x88: 'I',
    0x89: 'J',
    0x8A: 'K',
    0x8B: 'L',
    0x8C: 'M',
    0x8D: 'N',
    0x8E: 'O',
    0x8F: 'P',
    0x90: 'Q',
    0x91: 'R',
    0x92: 'S',
    0x93: 'T',
    0x94: 'U',
    0x95: 'V',
    0x96: 'W',
    0x97: 'X',
    0x98: 'Y',
    0x99: 'Z',
    0x9A: '(',
}

In [95]:
tile_bytes = [i for i in range(0x80, 0x9A)]
uppercase_letters = [chr(i) for i in range(ord('A'), ord('Z')+1)]
lowercase_tile_bytes = [i for i in range(0xA0, 0xB9+1)]
print('len lowercase tile bytes: ')
lowercase_letters = [chr(i) for i in range(ord('a'), ord('z')+1)]
uppercase_map = dict(zip(tile_bytes, uppercase_letters))
lowercase_map = dict(zip(lowercase_tile_bytes, lowercase_letters))

byte_to_letter_map = {**uppercase_map, **lowercase_map}

len lowercase tile bytes: 


In [73]:
print(byte_to_letter_map)

{128: 'A', 129: 'B', 130: 'C', 131: 'D', 132: 'E', 133: 'F', 134: 'G', 135: 'H', 136: 'I', 137: 'J', 138: 'K', 139: 'L', 140: 'M', 141: 'N', 142: 'O', 143: 'P', 144: 'Q', 145: 'R', 146: 'S', 147: 'T', 148: 'U', 149: 'V', 150: 'W', 151: 'X', 152: 'Y', 153: 'Z', 160: 'a', 161: 'b', 162: 'c', 163: 'd', 164: 'e', 165: 'f', 166: 'g', 167: 'h', 168: 'i', 169: 'j', 170: 'k', 171: 'l', 172: 'm', 173: 'n', 174: 'o', 175: 'p', 176: 'q', 177: 'r', 178: 's', 179: 't', 180: 'u', 181: 'v', 182: 'w', 183: 'x', 184: 'y', 185: 'z'}


In [80]:
import numpy as np
game_area_tiles = []
game_area = [[304, 313, 304, 313, 277, 312, 274, 274, 274, 274, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [313, 304, 313, 304, 277, 278, 279, 279, 279, 279, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [304, 304, 304, 304, 293, 294, 266, 266, 290, 303, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [304, 304, 304, 304, 271, 290, 290, 290, 290, 290, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [304, 304, 326, 327, 271, 266, 266, 266, 266, 266, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [304, 304, 342, 343, 334, 282, 282, 282, 282, 282, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [304, 313, 304, 313, 304, 313, 304, 313, 304, 313, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [313, 304, 313, 304, 313, 304, 313, 304, 0, 1, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [304, 313, 304, 313, 304, 313, 304, 313, 2, 3, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [313, 304, 313, 304, 313, 304, 313, 304, 40, 41, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [378, 378, 378, 378, 378, 378, 378, 378, 42, 43, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [383, 383, 383, 383, 383, 383, 383, 383, 383, 380, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [160, 178, 383, 171, 168, 181, 164, 163, 383, 380, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [383, 383, 383, 383, 383, 383, 383, 383, 383, 380, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [163, 164, 162, 160, 163, 164, 178, 232, 383, 380, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [378, 378, 378, 378, 378, 378, 378, 378, 378, 382, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383], [383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383, 383]]
for lst in game_area:
    for tile in lst:
        game_area_tiles.append(tile)
numbers = [83, 83, 83, 83, 77, 48, 57, 48, 57, 48, 57, 48, 57, 5, 6, 83, 83, 83, 83, 18, 18, 18, 18, 18, 90, 57, 48, 57, 48, 57, 48, 57, 48, 21, 56, 18, 18, 18, 18, 18, 18, 18, 18, 18, 90, 48, 57, 48, 57, 48, 57, 48, 57, 21, 56, 18, 18, 18, 18, 23, 23, 23, 23, 23, 93, 57, 48, 57, 48, 57, 48, 57, 48, 21, 22, 23, 23, 23, 23, 75, 10, 10, 10, 10, 31, 48, 57, 48, 57, 48, 48, 48, 48, 37, 38, 10, 10, 34, 47, 75, 75, 75, 75, 75, 31, 57, 48, 57, 48, 48, 48, 48, 48, 15, 34, 34, 34, 34, 34, 10, 10, 75, 75, 75, 31, 48, 57, 48, 57, 48, 48, 70, 71, 15, 10, 10, 10, 10, 10, 26, 26, 26, 26, 26, 79, 57, 48, 57, 48, 48, 48, 86, 87, 78, 26, 26, 26, 26, 26, 48, 57, 48, 48, 48, 48, 48, 57, 48, 57, 48, 57, 48, 57, 48, 57, 48, 57, 48, 57, 57, 48, 48, 48, 48, 48, 57, 48, 57, 48, 57, 48, 57, 48, 57, 48, 57, 48, 57, 48, 48, 57, 48, 48, 70, 71, 48, 57, 48, 57, 48, 57, 48, 57, 48, 57, 48, 57, 48, 57, 57, 48, 48, 48, 86, 87, 57, 48, 57, 48, 57, 48, 57, 48, 57, 48, 57, 48, 57, 48, 121, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 123, 124, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 124, 124, 130, 136, 141, 141, 128, 129, 128, 145, 127, 134, 152, 140, 189, 127, 127, 127, 127, 127, 124, 124, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 124, 124, 129, 139, 128, 136, 141, 132, 127, 168, 178, 127, 160, 173, 127, 174, 163, 163, 127, 238, 124, 125, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 126, 83]
numbers2 = [122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 123, 124,
127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 124, 124,
130, 136, 141, 141, 128, 129, 128, 145, 127, 134, 152, 140, 189, 127, 127, 127, 127, 127, 124, 124,
127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 127, 124, 124,
129, 139, 128, 136, 141, 132, 127, 168, 178, 127, 160, 173, 127, 174, 163, 163, 127, 127, 124, 125,
122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 122, 126, 83]
characters = [byte_to_letter_map.get(number, '?') for number in numbers]
characters2 = [byte_to_letter_map.get(number, '?') for number in numbers2]
for i in characters2:
    if i != '?':
        print(i)

C
I
N
N
A
B
A
R
G
Y
M
B
L
A
I
N
E
i
s
a
n
o
d
d


In [50]:
hex_values = ["0x80", "0xb9"]
base_10_values = [int(value, 16) for value in hex_values]
print(base_10_values)

[128, 185]


In [61]:
for i in numbers2:
    if i > 127:
        print(i)
print('other list: ')
# for i in numbers:
#     if i > 127:
#         print(i)

134
152
140
189
160
173
174
163
163
238
other list: 
130
136
141
141
128
129
128
145
134
152
140
189
129
139
128
136
141
132
168
178
160
173
174
163
163
238


In [24]:
from pyboy import PyBoy
pyboy = PyBoy('PokemonRed.gb')
for _ in range(6000):
    pyboy.tick()

SyntaxError: invalid syntax (3723679755.py, line 2)

In [28]:
0xC506 > 22687

True